# DSPN FINAL PROJECT
## Micheal Olorunsola

# **Title**: Frequencies & Feelings: Exploring Music Habits and Mental Health
* * *

# **Background**

Music is one of the most universal human behaviors, and the manner in which one engages with it, more often than not, can reveal much about their state of being. In the wake of personal hardship or unspoken sadness, some listeners cling to a favorite genre for years; others, in search of something fleeting and new, chase the thrill of constant discovery. Some put headphones on the moment work begins; others save music for the commute home. These habits feel deeply personal, and for good reason: there's growing evidence that music is not merely entertainment but a genuine emotional and psychological tool that people reach for in moments of stress, sadness, or joy. Yet, the relationship between *how* people consume music and their broader mental health remains underexplored. As such, this project uses a public survey dataset to investigate whether daily listening habits, genre preferences, and exploratory behavior correlate with self-reported severity of common mental health conditions like anxiety, depression, and OCD. 
* * *

In [24]:
music.df <- read.csv("mxmh_survey_results.csv")

# **Variables**
As mentioned, this dataset was collected via a public online survey and contains 736 observations across 33 variables:
+ Timestamp: datetime, when the survey was submitted, not analytically relevant
+ Age: continuous, ranges 10 to 89
+ Primary streaming service: categorical (Spotify, Pandora, etc.)
+ Hours per day: continuous, 0 to 24
+ While working: binary (Yes/No), whether they listen during work or study
+ Instrumentalist: binary (Yes/No), whether the respondent plays an instrument
+ Composer: binary (Yes/No), whether the respondent composes music
+ Fav genre: categorical, 16 possible genres
+ Exploratory: binary (Yes/No), whether they actively seek out new music
+ Foreign languages: binary (Yes/No), whether they listen to music in foreign languages
+ BPM: continuous, self-reported average beats per minute of music they listen to
+ Frequency [Genre]: 16 columns, one per genre, ordinal (Never, Rarely, Sometimes, Very Frequently)
+ Anxiety, Depression, Insomnia, OCD: each an integer from 0 to 10, self-reported severity
+ Music effects: categorical (Improve, No effect, Worsen)
+ Permissions: consent acknowledgment, not analytically relevant
* * * 

# **Hypotheses**

This analysis is organized around two central research questions each with specific hypotheses about how music listening behavior and genre preferences relate to mental health outcomes.

## Research Question 1: *Listening Habits and Mental Health*

The first set of hypotheses examines whether daily listening behavior predicts mental health outcomes, where **Y** varies by hypothesis and **X** captures different dimensions of listening behavior:

+ **H1**: Higher daily listening hours are associated with higher mental health severity scores. Model: **Y** = f(`Hours.per.day`), evaluated using non-parametric regression (*spline*).
+ **H2**: Listening while working is associated with different mental health severity scores compared to those who do not. Model form: **Y** = f(`While.working`),  evaluated using *bootstrapped mean differences* between listeners who do and do not listen while working, for each outcome.
+ **H3**: Exploratory listeners show different mental health severity profiles than non-exploratory listeners. Model form: **Y** = f(`Exploratory`), evaluated using *bootstrapped mean differences* between exploratory and non-exploratory listeners, for each outcome.
+ **H4**: Daily listening hours are associated with whether a respondent perceives music as improving their mental health or not. Model form: Music.effects = f(`Hours.per.day`), evaluated using *kNN*.

## Research Question 2: *Genre Preferences and Mental Health*

The second set of hypotheses examines whether genre preference and listening frequency predict mental health outcomes and perceived music effects:

+ **H5**: Certain favorite genres are associated with higher or lower mental health severity scores. Model form: **Y** = f(`Fav.genre`), evaluated using a *permutation test* to assess whether the observed variation in mental health scores across genre groups is greater than what would be expected by chance.
* * *

# **Data Organization**

Because our dataset is a single table with one row per survey respondent (and 33 columns), no joins or reshaping will be needed going into analysis. However, prior to fitting any models, a few cleansing steps are necessary. 

`BPM` is the most problematic column, with 107 missing values and several clearly invlaid entries (including a value of 999,999,999 and multiple values near zero), so it will be dropped entirely since it is not central to any hypothesis. Most other columns have only a handful of missing values, and those rows will simply be excluded list-wise for any model that uses them, with the exception of `Music.effects` which has 8 missing values that will be excluded specifically for **H5**. We also note that `Timestamp` and `Permissions` carry no analytical value and will be dropped. Finally, given the *severe* class imbalance in `Music.effects` (542 Improve, 169 No effect, 17 Worsen), we collapse Worsen and No effect into a single **No Improve** category to protect against small sample sizes and make the classification task more tractable.

Following these steps, we produce a much cleaner dataset: each row is one respondent, each column is one variable, and the four mental health outcomes are consistently scored 0 to 10. 

An example of the final cleaned table going into my analysis is produced below:

In [25]:
library(tidyverse)


music.df <- music.df |>
    select(-Timestamp, -Permissions, -BPM) |>
    mutate(Music.effects = ifelse(Music.effects == "Improve", "Improve", "No Improve"))

head(music.df)

,Age,Primary.streaming.service,Hours.per.day,While.working,Instrumentalist,Composer,Fav.genre,Exploratory,Foreign.languages,Frequency..Classical.,⋯,Frequency..Pop.,Frequency..R.B.,Frequency..Rap.,Frequency..Rock.,Frequency..Video.game.music.,Anxiety,Depression,Insomnia,OCD,Music.effects
,<int>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
1,18,Spotify,3.0,Yes,Yes,Yes,Latin,Yes,Yes,Rarely,⋯,Very frequently,Sometimes,Very frequently,Never,Sometimes,3,0,1,0,No Improve
2,63,Pandora,1.5,Yes,No,No,Rock,Yes,No,Sometimes,⋯,Sometimes,Sometimes,Rarely,Very frequently,Rarely,7,2,2,1,No Improve
3,18,Spotify,4.0,No,No,No,Video game music,No,Yes,Never,⋯,Rarely,Never,Rarely,Rarely,Very frequently,7,7,10,2,No Improve
4,61,YouTube Music,2.5,Yes,No,Yes,Jazz,Yes,Yes,Sometimes,⋯,Sometimes,Sometimes,Never,Never,Never,9,7,3,3,Improve
5,18,Spotify,4.0,Yes,No,No,R&B,Yes,No,Never,⋯,Sometimes,Very frequently,Very frequently,Never,Rarely,7,2,5,9,Improve
6,18,Spotify,5.0,Yes,Yes,Yes,Jazz,Yes,Yes,Rarely,⋯,Very frequently,Very frequently,Very frequently,Very frequently,Never,8,8,7,7,Improve


* * *
# Analysis
Finally, with a clean dataset in hand, we now turn to the models outlined in the hypotheses section, working through each research question in order.

### H1: Hours per day vs. Mental outcome

For H1, we used a generalized additive model (GAM) with a spline term on `Hours per day`, allowing the relationship between daily listening time and each mental health outcome take whatever shape the data suggests.

In [26]:
library(mgcv)
h1.anxiety <- gam(Anxiety ~ s(Hours.per.day), data = music.df)
h1.depression <- gam(Depression ~ s(Hours.per.day), data = music.df)
h1.insomnia <- gam(Insomnia ~ s(Hours.per.day), data = music.df)
h1.ocd <- gam(OCD ~ s(Hours.per.day), data = music.df)

h1.results <- data.frame(
  disorder   = c("Anxiety", "Depression", "Insomnia", "OCD"),
  R_squared = c(summary(h1.anxiety)$r.sq,    summary(h1.depression)$r.sq,
                summary(h1.insomnia)$r.sq,   summary(h1.ocd)$r.sq),
  P_value   = c(summary(h1.anxiety)$s.table[,"p-value"],
                summary(h1.depression)$s.table[,"p-value"],
                summary(h1.insomnia)$s.table[,"p-value"],
                summary(h1.ocd)$s.table[,"p-value"])
)

h1.results

disorder,R_squared,P_value
<chr>,<dbl>,<dbl>
Anxiety,0.005004403,0.1526485217
Depression,0.024485331,0.0017713493
Insomnia,0.021385605,0.0007629738
OCD,0.014989355,0.0036074410


The results are mixed, but interesting; `Hours per day` comes out non-significant for predicting `Anxiety` (p = 0.153), suggesting daily listening time doesn't meaningfully track with how anxious someone reports feeling. On the other hand, `Depression`, `Insomnia`, and `OCD` all show statistically significant smooth terms (p < 0.01 for all three), meaning there is some real non-linear relationship between hours listened and those outcomes. That said, the explained deviance is quite low across the board; $R^2$ values range from about (0.01 to 0.025) so `Hours per day alone` is not a strong predictor of any single outcome. It's a signal, but a modest one.

### H2: While working vs. Mental outcome

For H2, we're bootstrapping the mean difference in each mental health outcome between respondents who listen while working and those who don't, using 10,000 resamples to build a 95% confidence interval on that difference. If zero falls outside the interval, we take that as evidence of a meaningful difference between the groups.

In [28]:
set.seed(403)
B <- 10000

library(boot)

yes   <- music.df |> filter(While.working == "Yes")
no    <- music.df |> filter(While.working == "No")

boot.diff <- function(outcome) {
  fn <- function(data, index) {
    mean(yes[[outcome]][sample(nrow(yes), length(index), replace = TRUE)]) -
    mean(no[[outcome]][sample(nrow(no), length(index), replace = TRUE)])
  }
  boot(music.df, fn, R = B)
}

h2.anxiety    <- boot.diff("Anxiety")
h2.depression <- boot.diff("Depression")
h2.insomnia   <- boot.diff("Insomnia")
h2.ocd        <- boot.diff("OCD")

# calculate confidence interval
conf_calc <- function(est, sd) {
  return(c(est - 1.96 * sd, est + 1.96 * sd))
}

h2.results <- data.frame(
  Outcome  = c("Anxiety", "Depression", "Insomnia", "OCD"),
  Obs_Diff = c(h2.anxiety$t0, h2.depression$t0, h2.insomnia$t0, h2.ocd$t0),
  CI_Low   = c(conf_calc(h2.anxiety$t0, sd(h2.anxiety$t))[1],
               conf_calc(h2.depression$t0, sd(h2.depression$t))[1],
               conf_calc(h2.insomnia$t0, sd(h2.insomnia$t))[1],
               conf_calc(h2.ocd$t0, sd(h2.ocd$t))[1]),
  CI_High  = c(conf_calc(h2.anxiety$t0, sd(h2.anxiety$t))[2],
               conf_calc(h2.depression$t0, sd(h2.depression$t))[2],
               conf_calc(h2.insomnia$t0, sd(h2.insomnia$t))[2],
               conf_calc(h2.ocd$t0, sd(h2.ocd$t))[2])
)

h2.results

Outcome,Obs_Diff,CI_Low,CI_High
<chr>,<dbl>,<dbl>,<dbl>
Anxiety,0.0937500,-0.19712644,0.3846264
Depression,0.2839674,-0.03118271,0.5991175
Insomnia,0.2826087,-0.02278140,0.5879988
OCD,0.5224185,0.23229299,0.8125440


The results here are largely null, with `Anxiety`, `Depression`, and `Insomnia` all producing confidence intervals that straddle zero, meaning we can't confidentrly attribute any difference in those outcomes to whether someone listens while working or not. `OCD` is the one exception, with an observed difference of 0.52 and a confidence interval of [0.23, 0.81] that sits above zero, but (in all honesty) this effect is modest and could even be presenting significant due to random seeding. Thus, it isn't plausible to conclude that listening while working has a meaningful relationship with any other four mental health outcomes we're examining here.

### H3: Exploratory listening vs Mental outcome

Now for H3, the setup is identical to H2—bootstrapped mean differences, same 10,000 resamples, same confidence interval logic. All we're doing is swapping `While.working` for `Exploratory` as our grouping variable:

In [ ]:
set.seed(403)
explorer <- music.df |> filter(Exploratory == "Yes")
nonexplorer <- music.df |> filter(Exploratory == "No")

boot.diff.exp <- function(outcome) {
  fn <- function(data, index) {
    mean(explorer[[outcome]][sample(nrow(explorer), length(index), replace = TRUE)]) -
    mean(nonexplorer[[outcome]][sample(nrow(nonexplorer), length(index), replace = TRUE)])
  }
  boot(music.df, fn, R = 10000)
}

h3.anxiety    <- boot.diff.exp("Anxiety")
h3.depression <- boot.diff.exp("Depression")
h3.insomnia   <- boot.diff.exp("Insomnia")
h3.ocd        <- boot.diff.exp("OCD")

h3.results <- data.frame(
  Outcome  = c("Anxiety", "Depression", "Insomnia", "OCD"),
  Obs_Diff = c(h3.anxiety$t0, h3.depression$t0,
               h3.insomnia$t0, h3.ocd$t0),
  CI_Low   = c(conf_calc(h3.anxiety$t0, sd(h3.anxiety$t))[1],
               conf_calc(h3.depression$t0, sd(h3.depression$t))[1],
               conf_calc(h3.insomnia$t0, sd(h3.insomnia$t))[1],
               conf_calc(h3.ocd$t0, sd(h3.ocd$t))[1]),
  CI_High  = c(conf_calc(h3.anxiety$t0, sd(h3.anxiety$t))[2],
               conf_calc(h3.depression$t0, sd(h3.depression$t))[2],
               conf_calc(h3.insomnia$t0, sd(h3.insomnia$t))[2],
               conf_calc(h3.ocd$t0, sd(h3.ocd$t))[2])
)

h3.results

Interesting results here. Unlike H2, three out of four outcomes come back significant. `Anxiety` is the only one whose confidence interval straddles zero [-0.14, 0.45], so we can't draw any conclusions there. `Depression`, `Insomnia`, and `OCD` however all produce intervals that sit entirely above zero, with observed differences of 0.61, 0.70, and 0.48 respectively, meaning exploratory listeners consistently report higher severity scores across those three conditions. This isn't entirely surprising as people who constantly seek out new music sometimes do so as a way to cope with, process, or escape a difficult reality. The data can't tell us which direction the relationship runs (whether music exploration is a *response* to poor mental health condition rather than a *cause*; willing to bet that it's a response but that's neither here nor there), but the pattern is consistent enough to be worth noting.

### H4: Hours per day vs. Music effect

For H4 we're using kNN to classify whether a respondent perceives music as improving their mental health or not, using Hours per day as the predictor. We produce a 80/20 train/test split and fit the model with k = 5, reporting accuracy and a confusion matrix to evaluate performance.

In [ ]:
library(class)
set.seed(403)
i <- sample(1:nrow(music.df), size = floor(0.8 * nrow(music.df)))

train.x <- matrix(music.df$Hours.per.day[i], ncol = 1)
test.x  <- matrix(music.df$Hours.per.day[-i], ncol = 1)
train.y <- music.df$Music.effects[i]
test.y  <- music.df$Music.effects[-i]

h4.pred <- knn(train.x, test.x, train.y, k = 5)

mean(h4.pred == test.y)
table(Predicted = h4.pred, Actual = test.y)

"Based on our summary confusion matrix and accuracy, we see that while the model achieves 76% accuracy, this is largely illusory; the model predicts Improve for nearly every observation, catching only 1 of 36 No Improve cases correctly (the classifier is clearly defaulting to the majority class). This suggests that `Hours.per.day` alone carries little meaningful signal for distinguishing whether someone perceives music as beneficial to their mental health or not.

### H5: Genre preference vs Mental outcome

For H5 we're shifting to a permutation test to assess whether the variation in mental health scores across favorite genre groups is greater than what we'd expect by chance; if a genre has no relationship with mental health outcomes, then randomly shuffling genre labels across respondents shouldn't change much. We run that shuffle thousands of times to build a null distribution, then see where our observed statistic lands within it.

In [ ]:
set.seed(403)

n_perm <- 10000
outcomes <- c("Anxiety", "Depression", "Insomnia", "OCD")

perm_test <- function(outcome) {
  observed <- var(tapply(music.df[[outcome]], music.df$Fav.genre, mean))
  null_dist <- replicate(n_perm, {
    shuffled <- sample(music.df$Fav.genre)
    var(tapply(music.df[[outcome]], shuffled, mean))
  })
  p_value <- mean(null_dist >= observed)
  data.frame(Outcome = outcome, Observed_Var = observed, P_Value = p_value)
}

h4.results <- rbind(
  perm_test("Anxiety"),
  perm_test("Depression"),
  perm_test("Insomnia"),
  perm_test("OCD")
)

h4.results

None of the four outcomes come back significant at a conventional threshold. Specifically, `Anxiety`, `Insomnia`, and `OCD` all have p-values well above 0.05, and `Depression` is the closest at 0.10 but still doesn't clear the bar. The permutation test is telling us that the variation in mean mental health scores across genre groups isn't meaningfully greater than what you'd expect from random shuffling alone, so we **can't conclude that favorite genre has a reliable relationship with any of the four outcomes**. As such, with 16 genre categories and some very small group sizes (e.g, Gospel at 6, Latin at 3, Lofi at 10), there simply isn't enough data in certain cells to detect a real signal even if one exists.
* * *

# **Conclusions**
Across the five hypotheses tested, the results paint a modest but interesting picture of the relationship between music listening behavior, genre preferences, and mental health. The GAM models for H1 suggested that daily listening hours have a small but statistically significant non-linear relationship with `Depression`, `Insomnia`, and `OCD`, though the low explained deviance across all four outcomes makes it difficult to draw strong conclusions. The bootstrapped analyses for H2 and H3 were similarly mixed; listening while working showed no meaningful relationship with any outcome, while exploratory listening tracked consistently with higher `Depression`, `Insomnia`, and `OCD` scores, lending some support to the idea that people struggling most with their mental health are the ones reaching most actively for music as an outlet. The kNN classifier for H4 achieved 76% accuracy but was largely predicting the majority class, offering little real insight into what separates those who perceive music as helpful from those who don't. The permutation test for H5 returned null results across all four outcomes, suggesting that favorite genre alone does not reliably predict mental health severity, likely hampered by small group sizes in several genre categories. Taken together, the findings suggest that while music listening behavior does carry some signal with respect to mental health, the relationship is likely more nuanced than any one model here can full capture.
* * *